# Activations & Gradients, BatchNorm（详细中文注释版）

Karpathy《building makemore **Part 3**: Activations & Gradients, BatchNorm》跟练笔记本。

这一讲不再是「把 MLP 搭出来」，而是**深入神经网络内部去看**：
- **激活值（activations）** 分布健不健康（有没有饱和、有没有死掉）；
- **梯度（gradients）** 有没有在深层网络里消失/爆炸；
- **初始化（initialization）** 该怎么缩放权重，让第一步 loss 就正常；
- **BatchNorm（批归一化）** 怎么用「强行把每层输入拉回标准正态」来让深层网络稳定可训练。

---

### ⚠️ 运行前必看（关于「别让电脑卡死」）

按你的要求，**本文件只加注释、不改你写的代码**。但你原来的代码里有几处 bug，其中两处会让电脑**死循环卡死**，我在对应 cell 前都用 `> ⚠️` 醒目标了出来：

| 位置 | bug | 直接运行的后果 | 正确写法 |
|---|---|---|---|
| 训练 cell | `while torch.no_grad():` | **死循环，电脑卡死** | `with torch.no_grad():` |
| 统计 cell | `while torch.no_grad():` | **死循环，电脑卡死** | `with torch.no_grad():` |
| 参数 cell | `b1`、`bnstd_running` 放进了 `parameters` | 更新时 `-lr*p.grad` 里 `p.grad=None` → 报 `TypeError` 崩溃 | 从 `parameters` 里去掉这两个 |

**所以：带 `> ⚠️` 标记的 cell 先别直接运行**（要跑就照注释里的正确写法改一个词）。
想要「能一键跑通、又不会卡机」的版本，请打开我另外新建的 **`Activations_BatchNorm_生产案例.ipynb`**——那是全新写的、安全可跑的代码。

## 0. 导入库

和前几讲一样：`torch` 主库、`nn`/`F` 里的现成组件、`matplotlib` 画图（本讲会画大量分布直方图）。

In [ ]:
import torch                       # PyTorch 主库(张量 + 自动求导)
import torch.nn as nn              # 神经网络模块(本讲主要自己手搓,少量用到)
import torch.nn.functional as F    # 函数式接口:cross_entropy / softmax 等
import  matplotlib.pyplot as plt   # 画图(本讲用来画激活/梯度分布直方图)
%matplotlib inline                 # 让图直接嵌在 notebook 里显示

## 1. 读数据 + 建词表

和前两讲完全一样：读入 32033 个名字，建立字符↔编号的双向映射。

In [ ]:
words=open('names.txt','r').read().splitlines()  # 读 names.txt,按行切成名字列表(约 32033 个)
words[:8]                                        # 看前 8 个,确认读对了

In [ ]:
chars=sorted(list(set(''.join(words))))          # 所有出现过的字符,去重排序(a..z 共 26 个)
stoi={s:i+1 for i,s in enumerate(chars)}         # 字符->编号: a=1,b=2,...,z=26
stoi['.']=0                                      # '.' 作为起始/结束特殊符,编号 0
itos={i:s for s,i in stoi.items()}               # 反过来:编号->字符,采样时用
vocab_size=len(itos)                             # 词表大小 = 27
block_size=3                                     # 上下文长度:用前 3 个字符预测下一个

## 2. 构造数据集（滑动窗口）

把每个名字用「前 3 个字符 → 第 4 个字符」的滑动窗口切成很多 (X, Y) 样本。封装成函数，方便对 训练/验证/测试 三个集合分别调用。

In [ ]:
def build_dataset(words):                         # 输入名字列表 -> 输出 (X, Y)
    X,Y=[],[]                                     # X=上下文(每个3个编号), Y=目标字符编号
    for w in words:                               # 逐个名字
        context=[0]*block_size                    # 上下文初始化为 [0,0,0](全是 '.')
        for ch in w+'.':                          # 遍历名字每个字符,末尾补 '.' 表示结束
            ix=stoi[ch]                           # 当前字符编号
            X.append(context)                     # 记录:当前上下文 ->
            Y.append(ix)                          #        当前字符(目标)
            context=context[1:]+[ix]              # 窗口右移一格:丢掉最左,补上当前字符
    X=torch.tensor(X)                             # list -> 张量,形状 (样本数, 3)
    Y=torch.tensor(Y)                             # list -> 张量,形状 (样本数,)
    return X,Y

## 3. 切分 训练/验证/测试 集（80% / 10% / 10%）

In [ ]:
import  random                                  # 打乱名字顺序用
random.seed(42)                                  # 固定随机种子,保证每次切分一致
random.shuffle(words)                            # 原地打乱名字列表
n1=int(0.8*len(words))                           # 前 80% 的分界点
n2=int(0.9*len(words))                           # 前 90% 的分界点
Xtr,Ytr=build_dataset(words[:n1])                # 训练集(80%)
Xdev,Ydev=build_dataset(words[n1:n2])            # 验证集(10%),调超参用
Xte,Yte=build_dataset(words[n2:])                # 测试集(10%),最后才看一次

---

# 第一部分：手搓一层带 BatchNorm 的 MLP（cell 5~9）

这一段用**裸张量**手动实现「Embedding → Linear → BatchNorm → Tanh → 输出」，
重点体会 **初始化缩放** 和 **BatchNorm** 到底在干嘛。

> 这一部分和第二部分(cell 10 之后的模块化版本)是**两个独立的小实验**，各自 `parameters` 不同，互不影响。

### 3.1 初始化参数

两个关键点：
1. **W1 的缩放** `*(5/3)/sqrt(fan_in)`：这是 tanh 的 **Kaiming 初始化**。`5/3` 是 tanh 的增益(gain)，`sqrt(fan_in)` 让每层输出方差保持 ~1，避免越深越饱和。
2. **W2、b2 缩小**（`*0.01`、`*0`）：让**第一步的 logits 接近 0**，初始 loss ≈ `ln(27)≈3.29` 而不是几十——避免开局「浪费好几百步把过大的 loss 压下来」（hockey-stick 曲线）。

> ⚠️ **这一格有 bug（但按你的要求不改）**：`parameters` 里放了 `b1` 和 `bnstd_running`。
> - `b1`：因为后面用了 BatchNorm，减均值时会把偏置整体抵消，`b1` **完全多余**，而且 cell 6 的前向根本没用到它 → 它 `grad` 恒为 `None`。
> - `bnstd_running`：它是**滑动统计缓冲区**（推理时用），不该被梯度更新。
>
> 结果：cell 6 更新时 `p.data += -lr*p.grad` 遇到 `p.grad=None` 会 **`TypeError` 崩溃**。
> **正确写法**：`parameters=[C,W1,W2,b2,bngain,bnbias]`（去掉 `b1` 和 `bnstd_running`），且 `bngain/bnbias` 才是 BN 里要训练的参数。

In [ ]:
n_embd=10                                         # 每个字符嵌入成 10 维向量
n_hidden=200                                      # 隐藏层神经元个数
g=torch.Generator().manual_seed(2147483647)       # 固定随机种子,保证可复现
C=torch.randn((vocab_size,n_embd),generator=g)    # 嵌入表 (27, 10)
W1=torch.randn((n_embd*block_size),n_hidden,generator=g)* (5/3)/((n_embd * block_size)**0.5)  # 第一层权重 (30,200),tanh 的 Kaiming 初始化:*(5/3)/sqrt(fan_in)
b1=torch.randn(n_hidden,generator=g)* 0.01        # 第一层偏置(有 BN 后其实多余,见上方 ⚠️)
W2=torch.randn((n_hidden,vocab_size),generator=g)* 0.01  # 输出层权重 (200,27),*0.01 缩小 -> 初始 logits 接近 0
b2=torch.randn(vocab_size,generator=g)* 0         # 输出层偏置,*0 = 初始化为全 0
bngain=torch.ones((1,n_hidden))                   # BatchNorm 的缩放 gamma,初始 1(可训练)
bnbias=torch.zeros((1,n_hidden))                  # BatchNorm 的平移 beta,初始 0(可训练)
bnstd_running=torch.ones((1,n_hidden))            # 推理用的滑动 std(缓冲区,不该训练,见上方 ⚠️)
bnmean_running = torch.zeros((1, n_hidden))       # 推理用的滑动 mean(缓冲区,不该训练)
parameters=[C,W1,W2,b1,b2,bngain,bnbias,bnstd_running]  # ⚠️ b1/bnstd_running 不该在此,正确应为 [C,W1,W2,b2,bngain,bnbias]
for p in parameters:p.requires_grad=True          # 开启梯度追踪(训练前必须)

### 3.2 训练循环（含手写 BatchNorm 前向 + 滑动统计）

BatchNorm 的核心三步（在 `hpreact` 上）：
1. 沿 **batch 维(dim=0)** 求这一批的均值 `bnmeani` 和标准差 `bnstdi`；
2. 归一化 `(hpreact - mean)/std`，再用可训练的 `bngain/bnbias` 缩放平移；
3. 用**滑动平均**把每一批的 mean/std 累积进 `bnmean_running/bnstd_running`，供推理时用（推理时没有 batch，不能现算）。

> ⚠️⚠️ **这一格有会「卡死电脑」的 bug（按你要求不改）——先别直接运行！**
> 第 12 行的 `while torch.no_grad():` 应该是 `with torch.no_grad():`。
> `torch.no_grad()` 每次都返回一个「真值」对象，`while` 判定永远为真 → **死循环，电脑卡死**。
> **要跑的话，先把这一个词 `while` 改成 `with`**，并且按 cell 5 的 ⚠️ 修好 `parameters`，否则更新处还会崩。

In [ ]:
max_steps=200                                     # 训练步数(这里只 200 步,很快,不会卡)
bath_size=32                                      # (原文拼写:应为 batch_size)小批量大小 32
lossi=[]                                           # 记录每步 loss(取 log10)
for i in range(max_steps):
    ix=torch.randint(0,Xtr.shape[0],(bath_size,),generator=g)  # 随机抽 32 个样本的下标
    Xb,Yb=Xtr[ix],Ytr[ix]                          # 这一批的输入 Xb 和目标 Yb
    emb=C[Xb]                                       # 查嵌入表 -> (32, 3, 10)
    embcat=emb.view(emb.shape[0],-1)               # 拼平 3 个字符 -> (32, 30)
    hpreact=embcat@W1                              # 第一层线性(注意没加 b1,因为 BN 会抵消)
    bnmeani=hpreact.mean(0,keepdim=True)           # 沿 batch 维求这一批的均值 (1,200)
    bnstdi=hpreact.std(0,keepdim=True)             # 沿 batch 维求这一批的标准差 (1,200)
    hpreact=bngain*(hpreact-bnmeani)/bnstdi+bnbias # BatchNorm:归一化后再 gamma 缩放 + beta 平移
    with torch.no_grad():                         # ⚠️BUG:应为 with!while 会死循环卡死电脑,运行前务必改成 with
        bnmean_running=0.999*bnmean_running+0.001*bnmeani  # 滑动更新推理用均值(EMA)
        bnstd_running=0.999*bnstd_running+0.001*bnstdi     # 滑动更新推理用标准差(EMA)
    h=torch.tanh(hpreact)                          # tanh 激活 -> (32, 200)
    logits=h@W2+b2                                 # 输出层 -> (32, 27) 的得分
    loss=F.cross_entropy(logits,Yb)                # 交叉熵损失(内部 = softmax + 负对数似然)
    for p in parameters:                           # 清零上一步的梯度
        p.grad=None
    loss.backward()                                # 反向传播,填充各参数的 .grad
    lr=0.1 if i <100 else 0.01                     # 简单的学习率衰减:前 100 步 0.1,之后 0.01
    for p in  parameters:                          # 手写梯度下降更新
        p.data+=-lr*p.grad                         # ⚠️ 若 parameters 含 b1/bnstd_running,这里 p.grad=None 会崩
    if i%10000==0:                                 # 偶尔打印一下进度
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())              # 记录 log10(loss)

### 3.3 画 loss 曲线

In [ ]:
plt.plot(lossi)                                   # loss(log10) 随步数变化,总体应下降

### 3.4 （可选）训练后一次性标定 BN 统计量

除了 cell 6 里的「滑动平均」，还有另一种更直接的做法：**训练完后，把整个训练集过一遍**，直接算出 `hpreact` 的均值/标准差当作推理统计量。Karpathy 证明这两种结果几乎一样。

> ⚠️⚠️ **同样的「卡死电脑」bug**：第 1 行 `while torch.no_grad():` 应为 `with torch.no_grad():`，否则**死循环卡死**。运行前先改这一个词。
> 另外这里算出的是局部变量 `bnmean/bnstd`，而 cell 9 用的是 cell 6 训练出来的 `bnmean_running/bnstd_running`，两者独立，本格可跑可不跑。

In [ ]:
while torch.no_grad():                            # ⚠️BUG:应为 with!否则死循环卡死电脑
    emb=C[Xtr]                                     # 整个训练集查嵌入
    embcat=emb.view(emb.shape[0],-1)              # 拼平 -> (N, 30)
    hpreact=embcat@W1                             # 第一层线性
    bnmean = hpreact.mean(0, keepdim=True)        # 整个训练集上的均值(一次性标定)
    bnstd = hpreact.std(0, keepdim=True)          # 整个训练集上的标准差

### 3.5 评估 训练/验证 loss

推理时**没有 batch**，所以 BN 用的是训练期间累积的 `bnmean_running/bnstd_running`（来自 cell 6），而不是现场算的 batch 统计量。这是 BN 训练/推理行为不同的关键点。

In [ ]:
@torch.no_grad()                                  # 评估不需要梯度,省内存也更快
def split_loss(split):
    x,y={                                          # 按名字选出对应的数据集
        'train':(Xtr,Ytr),
        'val':(Xdev,Ydev),
        'test':(Xte,Yte)
    }[split]
    emb=C[x]                                        # 查嵌入
    embcat=emb.view(emb.shape[0],-1)               # 拼平
    hpreact=embcat@W1                              # 第一层线性
    hpreact=bngain*(hpreact-bnmean_running)/bnstd_running+bnbias  # BN:推理用滑动统计量(不是现算的)
    h=torch.tanh(hpreact)                          # 激活
    logits=h@W2+b2                                 # 输出层
    loss=F.cross_entropy(logits,y)                 # 交叉熵
    print(split,loss.item())                       # 打印该集合的 loss
split_loss('train')                                # 训练集 loss
split_loss('val')                                  # 验证集 loss(和训练集接近说明没过拟合)

---

# 第二部分：把层封装成模块，搭 6 层深网络看「诊断图」（cell 10~18）

这一部分把 `Linear / BatchNorm1d / Tanh` 封装成**像 PyTorch 那样的类**，然后叠一个**很深(6 层)** 的网络，
用**四张诊断图**去看深层网络的健康状况：
1. 每层 **激活值** 分布（tanh 有没有饱和）；
2. 每层 **激活梯度** 分布；
3. 每个 **权重的梯度** 分布 + grad:data 比例；
4. **update:data 比例** 随训练变化（判断学习率合不合适，理想 ~1e-3）。

### 4.1 手写 Linear / BatchNorm1d / Tanh 模块

模仿 `torch.nn` 的接口：每个模块都有 `__call__`（前向）和 `parameters()`（返回可训练张量）。`BatchNorm1d` 内部用 `self.training` 区分训练/推理两种行为。

In [ ]:
class Linear:                                     # 全连接层:out = x @ W (+ b)
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight=torch.randn((fan_in,fan_out),generator=g)/fan_in**0.5  # Kaiming 初始化:除以 sqrt(fan_in) 保持方差
        self.bias=torch.zeros(fan_out) if bias else None                   # 偏置(可选;配 BN 时通常 bias=False)
    def __call__(self, x):
        self.out=x@self.weight                     # 线性变换
        if self.bias is not None:self.out+=self.bias  # 加偏置(若有)
        return self.out
    def parameters(self):
        return [self.weight]+([] if self.bias is None else [self.bias])  # 返回可训练参数

class BatchNorm1d:                                # 批归一化(1 维)
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps=eps                               # 防止除 0 的小常数
        self.momentum=momentum                     # 滑动统计的动量
        self.training = True                       # 训练模式开关(推理前要置 False)
        self.gamma=torch.ones(dim)                 # 可训练缩放(初始 1)
        self.beta=torch.zeros(dim)                 # 可训练平移(初始 0)
        self.running_mean = torch.zeros(dim)       # 推理用滑动均值(缓冲区)
        self.running_var = torch.ones(dim)         # 推理用滑动方差(缓冲区)
    def __call__(self, x):
        if self.training:                          # 训练:用当前 batch 的统计量
            xmean = x.mean(0, keepdim=True)        # batch 均值
            xvar=x.var(0,keepdim=True)             # batch 方差
        else:                                      # 推理:用滑动统计量
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)  # 归一化到均值0方差1
        self.out = self.gamma * xhat + self.beta   # 再缩放平移
        if self.training:                          # 训练时顺便更新滑动统计量
             with torch.no_grad():                 # (这里正确用了 with;别更新到计算图里)
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum *xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]             # 只有 gamma/beta 可训练(running_* 是缓冲区)
class Tanh:                                        # 激活层
  def __call__(self, x):
    self.out = torch.tanh(x)                       # tanh 挤到 (-1,1)
    return self.out
  def parameters(self):
    return []                                      # 无可训练参数

### 4.2 搭一个 6 层深的网络

`Linear(..., bias=False)` + `BatchNorm1d`：有 BN 就不需要 Linear 的偏置了。
最后一层的 `gamma *= 0.1` 是为了让**初始 logits 更小、更「不自信」**，初始 loss 更接近理论值。
中间那句 `layer.weight *= 1.0 # 5/3` 是 Karpathy 留的**实验开关**：改成 `5/3` 就是给 tanh 加增益，可对比激活分布的变化。

In [ ]:
n_embd = 10 # the dimensionality of the character embedding vectors   # 嵌入维度
n_hidden = 100 # the number of neurons in the hidden layer of the MLP  # 每个隐藏层宽度
g = torch.Generator().manual_seed(2147483647) # for reproducibility    # 固定种子

C = torch.randn((vocab_size, n_embd),            generator=g)          # 嵌入表 (27,10)
layers = [                                                             # 6 层:每层 Linear+BN+Tanh,最后一层只有 Linear+BN
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]


with torch.no_grad():                                                  # 初始化微调(不进计算图)
  # last layer: make less confident
  layers[-1].gamma *= 0.1                                              # 最后一层 BN 的 gamma 缩小 -> logits 更小更谦虚
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3                                         # 实验开关:改成 5/3 给 tanh 加增益,对比激活分布

parameters = [C] + [p for layer in layers for p in layer.parameters()]  # 汇总所有可训练参数
print(sum(p.nelement() for p in parameters)) # number of parameters in total  # 参数总量
for p in parameters:
  p.requires_grad = True                                              # 开启梯度追踪

### 4.3 训练（带 update:data 比例记录）

> ✅ **这一格是安全的，可以放心运行**：虽然 `max_steps=200000`，但结尾有 `if i >= 1000: break`，
> 实际只跑 **1001 步**（几秒钟），就是为了收集诊断用的统计量，**不会卡机**。
>
> `ud`（update-to-data ratio）记录每步「参数更新量的标准差 ÷ 参数本身的标准差」的 log10，理想值约 **1e-3**（即 -3），用来判断学习率是否合适。
>
> ⚠️ 如果你哪天想「真正训练到底」，把 `if i >= 1000: break` 删掉即可——但那会跑满 20 万步，**耗时几分钟到十几分钟**（不会卡死，只是慢），建议先在小步数上验证无误再放开。

In [ ]:
# same optimization as last time
max_steps = 200000                                                    # 名义步数(下面有 break,实际只跑 1001 步)
batch_size = 32                                                       # 小批量大小
lossi = []                                                            # 记录 loss
ud = []                                                               # 记录 update:data 比例(每步每参数)

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)     # 随机抽一批下标
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y                               # 这一批的 X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors                     # 查嵌入 (32,3,10)
  x = emb.view(emb.shape[0], -1) # concatenate the vectors            # 拼平 (32,30)
  for layer in layers:                                                # 依次过 6 层
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function                       # 交叉熵损失

  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph # 保留每层输出的梯度(为了画诊断图)
  for p in parameters:
    p.grad = None                                                     # 清零梯度
  loss.backward()                                                     # 反向传播

  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay          # 学习率衰减
  for p in parameters:
    p.data += -lr * p.grad                                            # 梯度下降更新

  # track stats
  if i % 10000 == 0: # print every once in a while                    # 偶尔打印进度
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())                                   # 记录 log10(loss)
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])  # 记录 update:data 比例

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization  # ✅ 只跑 1001 步就停,防卡机

### 诊断图 ①：每层激活值分布（tanh 有没有饱和）

看每个 Tanh 层输出的分布：`saturated` 是 `|激活|>0.97` 的比例。太高说明大量神经元饱和(梯度≈0，学不动)；分布应该比较「胖」而不是全挤在 ±1。

In [ ]:
plt.figure(figsize=(20, 4)) # width and height of the plot          # 宽 20 高 4 的画布
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer  # 遍历各层(去掉输出层)
  if isinstance(layer, Tanh):                                         # 只看 Tanh 层
    t = layer.out                                                     # 该层的激活值
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))  # 打印均值/标准差/饱和比例
    hy, hx = torch.histogram(t, density=True)                         # 算直方图
    plt.plot(hx[:-1].detach(), hy.detach())                           # 画分布曲线
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')                                  # 标题:激活分布

### 诊断图 ②：每层激活的梯度分布

理想情况：各层梯度分布应该**大致重合、宽度相近**。若越深越窄(消失)或越宽(爆炸)，说明网络不健康。

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad                                                # 该层激活的梯度(cell12 里 retain_grad 才有)
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')                                    # 标题:梯度分布

### 诊断图 ③：每个权重矩阵的梯度分布 + grad:data 比例

`grad:data ratio = grad.std()/data.std()`：衡量「这一步梯度相对权重本身有多大」。某个权重比例特别大，说明它更新得过猛，可能不稳。

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad                                                          # 该参数的梯度
  if p.ndim == 2:                                                     # 只看二维权重矩阵(跳过偏置/gamma/beta)
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))  # 打印 grad:data 比例
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');                          # 标题:权重梯度分布

### 诊断图 ④：update:data 比例随训练变化（判断学习率）

黑线画在 -3（即 1e-3）：各参数的曲线应大致落在这条线附近。**远高于**说明学习率太大、**远低于**说明太小/学得太慢。

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:                                                     # 只看二维权重
    plt.plot([ud[j][i] for j in range(len(ud))])                      # 画该参数每步的 update:data 比例
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot  # 参考线:1e-3
plt.legend(legends);

### 4.4 评估（切换到推理模式）

**关键**：评估前把每层 `training=False`，BatchNorm 才会改用 `running_mean/var`，而不是现场算 batch 统计量（推理时通常一次只有 1 个样本，没法算 batch 统计）。

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking          # 评估不追踪梯度
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)                                # 查嵌入
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd) # 拼平
  for layer in layers:                                                # 过所有层
    x = layer(x)
  loss = F.cross_entropy(x, y)                                        # 交叉熵
  print(split, loss.item())

# put layers into eval mode
for layer in layers:
  layer.training = False                                             # 关键:切到推理模式(BN 用 running 统计量)
split_loss('train')
split_loss('val')

### 4.5 从模型采样生成名字

从起始上下文 `[0,0,0]` 开始，每次预测下一个字符并右移窗口，直到采到 `.`(编号 0) 结束。

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)                    # 采样用的随机种子(+10 换一批结果)

for _ in range(20):                                                   # 生成 20 个名字

    out = []
    context = [0] * block_size # initialize with all ...              # 初始上下文 [0,0,0]
    while True:
      # forward pass the neural net
      emb = C[torch.tensor([context])] # (1,block_size,n_embd)        # 当前上下文的嵌入
      x = emb.view(emb.shape[0], -1) # concatenate the vectors        # 拼平
      for layer in layers:                                           # 过网络(此时应已 training=False)
        x = layer(x)
      logits = x
      probs = F.softmax(logits, dim=1)                               # 转成概率分布
      # sample from the distribution
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()  # 按概率采样下一个字符
      # shift the context window and track the samples
      context = context[1:] + [ix]                                   # 窗口右移
      out.append(ix)
      # if we sample the special '.' token, break
      if ix == 0:                                                    # 采到 '.' 就结束这个名字
        break

    print(''.join(itos[i] for i in out)) # decode and print the generated word  # 编号转回字符并打印

---

## 附录 A：BatchNorm 前向的交互式演示（滑块小工具）

拖动滑块改变第 0 个样本的值 `x0`，观察 BatchNorm 如何把一批数据(蓝点)减均值除标准差、映射成标准正态(红点)。**直觉**：BN 就是把任意分布强行「拉回」到均值 0、标准差 1。

> 需要 `ipywidgets` 和 `scipy`。若未安装：`pip install ipywidgets scipy`。运行安全、不会卡机。

In [ ]:
# BatchNorm forward pass as a widget

from ipywidgets import interact, interactive, fixed, interact_manual   # 交互滑块
import ipywidgets as widgets
import scipy.stats as stats                                            # 画正态曲线
import numpy as np

def normshow(x0):

  g = torch.Generator().manual_seed(2147483647+1)
  x = torch.randn(5, generator=g) * 5                                  # 5 个样本,乘 5 拉大方差
  x[0] = x0 # override the 0th example with the slider                 # 用滑块覆盖第 0 个样本
  mu = x.mean()                                                        # 这批的均值
  sig = x.std()                                                        # 这批的标准差
  y = (x - mu)/sig                                                     # BN 归一化结果

  plt.figure(figsize=(10, 5))
  # plot 0
  plt.plot([-6,6], [0,0], 'k')
  # plot the mean and std
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, mu, sig), 'b')                       # 蓝:输入分布
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, 0, 1), 'r')                          # 红:标准正态(BN 目标)
  # plot little lines connecting input and output
  for i in range(len(x)):
    plt.plot([x[i],y[i]], [1, 0], 'k', alpha=0.2)                      # 连线:输入 -> 输出
  # plot the input and output values
  plt.scatter(x.data, torch.ones_like(x).data, c='b', s=100)          # 蓝点:输入
  plt.scatter(y.data, torch.zeros_like(y).data, c='r', s=100)         # 红点:归一化输出
  plt.xlim(-6, 6)
  # title
  plt.title('input mu %.2f std %.2f' % (mu, sig))

interact(normshow, x0=(-30,30,0.5));                                   # 生成滑块(-30~30)

## 附录 B：纯 Linear 链的前向/反向统计

手动搭 `c = b @ a`，看前向激活的 std 和反向梯度的 std。用来理解「不做归一化时，方差会怎么随层数漂移」。

In [ ]:
# Linear: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

a = torch.randn((1000,1), requires_grad=True, generator=g)          # a.grad = b.T @ c.grad     # 输入向量
b = torch.randn((1000,1000), requires_grad=True, generator=g)       # b.grad = c.grad @ a.T     # 权重矩阵(未缩放)
c = b @ a                                                            # 前向:矩阵乘
loss = torch.randn(1000, generator=g) @ c                           # 随便造一个标量 loss
a.retain_grad()                                                     # 保留非叶子节点梯度以便观察
b.retain_grad()
c.retain_grad()
loss.backward()                                                    # 反向传播
print('a std:', a.std().item())                                    # 看前向各量的标准差
print('b std:', b.std().item())
print('c std:', c.std().item())                                    # 注意 c 的 std 被放大了(未除 sqrt(fan_in))
print('-----')
print('c grad std:', c.grad.std().item())                          # 看反向各量梯度的标准差
print('a grad std:', a.grad.std().item())
print('b grad std:', b.grad.std().item())

## 附录 C：Linear + BatchNorm 的前向/反向统计

和附录 B 对比：加了 BN 之后，前向输出 `out` 的 std 被稳定在 ~1，反向梯度也更「守规矩」——直观说明 BN 为什么能让深层网络更好训练。

In [ ]:
# Linear + BatchNorm: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

n = 1000
# linear layer ---
inp = torch.randn(n, requires_grad=True, generator=g)               # 输入
w = torch.randn((n, n), requires_grad=True, generator=g) # / n**0.5  # 权重(注释里的 /n**0.5 是可选的缩放对比)
x = w @ inp                                                         # 线性输出
# bn layer ---
xmean = x.mean()                                                   # 均值
xvar = x.var()                                                     # 方差
out = (x - xmean) / torch.sqrt(xvar + 1e-5)                        # BN 归一化 -> std≈1
# ----
loss = out @ torch.randn(n, generator=g)                           # 造一个标量 loss
inp.retain_grad()
x.retain_grad()
w.retain_grad()
out.retain_grad()
loss.backward()                                                    # 反向传播

print('inp std: ', inp.std().item())                               # 前向各量 std
print('w std: ', w.std().item())
print('x std: ', x.std().item())
print('out std: ', out.std().item())                               # BN 后 out 的 std 稳定在 ~1
print('------')
print('out grad std: ', out.grad.std().item())                     # 反向各量梯度 std
print('x grad std: ', x.grad.std().item())
print('w grad std: ', w.grad.std().item())
print('inp grad std: ', inp.grad.std().item())